In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
REPO_DIR = '/kaggle/working/semicon'
!git clone https://github.com/Ganesh-0509/semicon.git "{REPO_DIR}"

In [ ]:
import os

def find_gt_noisy(root):
    for r, dirs, files in os.walk(root):
        if 'GT' in dirs and 'NoisyLR' in dirs:
            return r
    return None

DATA_ROOT = find_gt_noisy('/kaggle/input')
assert DATA_ROOT is not None, 'GT/NoisyLR not found under /kaggle/input'
print('Using DATA_ROOT =', DATA_ROOT)

10-epoch sanity check of the new training setup vs. the width=32 baseline
(epoch-by-epoch from the finished 100-epoch run: 24.82 / 26.13 / 26.38 / 26.56 / 26.85 /
27.13 / 27.37 / 27.53 / 27.49 / 27.53 dB for epochs 1-10).
Changes under test: width=48 (1.69M params, up from 0.76M), EMA weight
averaging, random-crop augmentation, cosine warm-restart schedule, FFT +
multi-layer perceptual loss (FFT/multi-layer perceptual are on by default
in CombinedLoss, no flag needed).

In [ ]:
CKPT_DIR = '/kaggle/working/checkpoints'
!mkdir -p "{CKPT_DIR}"
!cd "{REPO_DIR}/src" && python train.py \
    --data_root "{DATA_ROOT}" \
    --out_dir "{CKPT_DIR}" \
    --epochs 10 \
    --batch_size 16 \
    --lr 2e-4 \
    --width 48 \
    --use_perceptual \
    --device cuda \
    --num_workers 2